In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import uvicorn

# Import the logic from your segregated files
from facial_recognition.model_loader import load_trained_model
from facial_recognition.embedding_extractor import generate_embedding
from facial_recognition.matcher import verify_identity

app = FastAPI(title="ArcFace Recognition Server")

# 1. Initialize the Model (Logic Brain)
# Ensure the path matches your face_model.dat or .h5 location
MODEL_PATH = "face_recognition/face_model.dat"
try:
    model = load_trained_model(MODEL_PATH)
    print("AI Model loaded successfully into memory.")
except Exception as e:
    print(f"Error loading model: {e}")
    model = None

# --- REQUEST SCHEMAS (Matches Flutter Data) ---

class EnrollmentRequest(BaseModel):
    image_base64: str

class VerificationRequest(BaseModel):
    current_image_base64: str
    stored_embedding: list[float]  # The 128-float list fetched from your DB

# --- API ENDPOINTS ---

@app.post("/register-face")
async def register_face(request: EnrollmentRequest):
    """
    Takes a face image, turns it into a vector, and returns it.
    The Backend/Flutter app should save this list in the database.
    """
    if model is None:
        raise HTTPException(status_code=500, detail="AI Model not initialized on server.")
    
    try:
        embedding = generate_embedding(model, request.image_base64)
        return {
            "status": "success",
            "face_vector": embedding  # Send this to your database
        }
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Processing error: {str(e)}")

@app.post("/verify-face")
async def verify_face(request: VerificationRequest):
    """
    Compares a live capture against a stored vector from the database.
    """
    if model is None:
        raise HTTPException(status_code=500, detail="AI Model not initialized on server.")
    
    try:
        # 1. Generate vector for the new image
        current_embedding = generate_embedding(model, request.current_image_base64)
        
        # 2. Compare with the stored vector using matcher.py logic
        result = verify_identity(current_embedding, request.stored_embedding, threshold=0.6)
        
        return {
            "status": "success",
            "match": result["verified"],
            "similarity_score": result["similarity"]
        }
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Verification error: {str(e)}")

# --- SERVER START ---
if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)